# Selector Group Chat - Ultra-Tiny Llama Version

`SelectorGroupChat` is a dynamic team-based chat system where participants take turns broadcasting messages to all other members. Unlike traditional round-robin approaches, it leverages a generative model to intelligently select the next speaker based on the ongoing conversation.

## 🚀 Why Ultra-Tiny Llama Models?

- **qwen2.5:0.5b (397MB)**: Lightning fast for planning and coordination
- **tinyllama:1.1b (637MB)**: Fast for specialized tasks
- **gemma2:2b (1.6GB)**: Fast for quality responses
- **All local** - no API costs or internet needed
- **Expected speed**: 5-20 seconds instead of minutes!

## 🎯 How SelectorGroupChat Works

1. **Intelligent Speaker Selection** – A model-driven approach determines the most relevant next speaker
2. **Configurable Roles & Descriptions** – Define unique roles and attributes for each participant
3. **Turn Management** – Prevents consecutive turns by the same speaker (optional)
4. **Flexible Selection Process** – Customize the selection prompt and override the model's decision

The system evaluates conversation history and participant attributes to select the most suitable next speaker.

In [ ]:
# Import required libraries for Llama SelectorGroupChat
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.messages import AgentEvent, ChatMessage
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.ollama import OllamaChatCompletionClient

print('✅ All libraries imported successfully!')
print('🚀 Ready to create ultra-tiny Llama SelectorGroupChat!')

In [ ]:
# Define utility tools for the agents
print('🔧 Setting up utility tools for the agents...')

def weather_check_tool(city: str) -> str:
    weather_data = {
        "Dubai": "Sunny, 35°C",
        "New York": "Cloudy, 22°C",
        "London": "Rainy, 18°C",
        "Tokyo": "Clear, 26°C"
    }
    return weather_data.get(city, "Weather data not available.")

def currency_exchange_tool(amount: float, from_currency: str, to_currency: str) -> str:
    exchange_rates = {
        ("USD", "EUR"): 0.92,
        ("EUR", "USD"): 1.08,
        ("USD", "AED"): 3.67,
        ("AED", "USD"): 0.27
    }
    rate = exchange_rates.get((from_currency, to_currency), None)
    if rate:
        converted_amount = amount * rate
        return f"{amount} {from_currency} is equal to {converted_amount:.2f} {to_currency}."
    return f"Exchange rate not available for {from_currency} to {to_currency}."

print('✅ Utility tools created successfully!')

In [ ]:
# Create ultra-tiny Llama model clients
print('🦙 Creating ultra-tiny Llama model clients...')
print('🎯 Each model has different strengths and thinking styles!')

# Model 1: Ultra-tiny, lightning fast for planning (397MB)
planning_model = OllamaChatCompletionClient(
    model='qwen2.5:0.5b'  # Ultra-tiny, only 397MB model
)

# Model 2: Super tiny, fast for weather tasks (637MB)
weather_model = OllamaChatCompletionClient(
    model='tinyllama:1.1b'  # Super tiny, only 637MB model
)

# Model 3: Tiny, fast for currency tasks (1.6GB)
currency_model = OllamaChatCompletionClient(
    model='gemma2:2b'  # Tiny, only 1.6GB model
)

print('✅ Three different ultra-tiny Llama models created!')
print('   - Planning model: qwen2.5:0.5b (397MB)')
print('   - Weather model: tinyllama:1.1b (637MB)')
print('   - Currency model: gemma2:2b (1.6GB)')

In [ ]:
# Create specialized agents with ultra-tiny Llama models
print('🤖 Creating specialized agents with ultra-tiny Llama models...')

# Planning Agent - coordinates and delegates tasks
planning_agent = AssistantAgent(
    name="PlanningAgent",
    description="An agent for planning tasks. It should break down tasks and delegate them to the appropriate agents.",
    model_client=planning_model,
    system_message="""
    You are a planning agent using qwen2.5:0.5b.
    Your job is to break down complex tasks into smaller, manageable subtasks.
    Your team members are:
        WeatherAgent: Checks weather conditions
        CurrencyAgent: Handles currency conversion
    
    When given a task, analyze it and delegate appropriate parts to your team members.
    Be concise and focused in your planning.
    """
)

# Create the Weather Agent
weather_agent = AssistantAgent(
    name="WeatherAgent",
    description="An agent that checks weather conditions for cities.",
    model_client=weather_model,
    system_message="""
    You are a weather agent using tinyllama:1.1b.
    Your job is to check weather conditions for cities using the weather_check_tool.
    When asked about weather, use the tool to get current conditions.
    Keep responses concise and focused on weather information.
    """
)

# Create the Currency Agent
currency_agent = AssistantAgent(
    name="CurrencyAgent",
    description="An agent that handles currency conversion.",
    model_client=currency_model,
    system_message="""
    You are a currency agent using gemma2:2b.
    Your job is to handle currency conversion using the currency_exchange_tool.
    When asked about currency conversion, use the tool to get exchange rates.
    Keep responses concise and focused on currency information.
    """
)

print('✅ Three specialized agents created successfully!')
print('   - PlanningAgent: Uses qwen2.5:0.5b for coordination')
print('   - WeatherAgent: Uses tinyllama:1.1b for weather tasks')
print('   - CurrencyAgent: Uses gemma2:2b for currency tasks')

In [ ]:
# Set up the SelectorGroupChat team with termination conditions
print('🔧 Setting up the SelectorGroupChat team...')

# Create termination conditions
max_msg_termination = MaxMessageTermination(max_messages=8)
text_termination = TextMentionTermination("COMPLETE")

# Combine conditions using OR (|)
combined_termination = max_msg_termination | text_termination

# Create the SelectorGroupChat team
selector_team = SelectorGroupChat(
    [planning_agent, weather_agent, currency_agent],
    model_client=planning_model,  # Uses planning model for speaker selection
    termination_condition=combined_termination,
    allow_repeated_speaker=False
)

print('✅ SelectorGroupChat team created successfully!')
print('   - Three specialized agents with different Llama models')
print('   - Intelligent speaker selection based on context')
print('   - Termination: Max 8 messages OR "COMPLETE" keyword')
print('   - Ready to test intelligent agent selection!')

In [ ]:
# Test the SelectorGroupChat team with a complex task
print('🧪 Testing the SelectorGroupChat team...')
print('📝 Task: Plan a business trip with weather and currency considerations')
print('⏱️  Expected time: 10-30 seconds')
print('🎯 Watch how the 3 different models collaborate intelligently!')

# Run the team
await Console(selector_team.run_stream(
    task="Plan a business trip from New York to Dubai. Check the weather in Dubai and convert $5000 USD to AED. Provide a complete travel plan."
))